# Do the drivers of finish time differ by profile?

Runs after `03_profiles.ipynb`. Reads `../data/clean.csv` and `../data/profiles.csv`.

`03` produced four profiles and showed they separate finish time by 51 minutes. That is a
difference in *level*. This notebook asks about *slope*: does an extra weekly mile, or another
month of running, buy the same thing for a novice as for a veteran?

Headline: no, and then yes. The raw comparison says the slopes differ a great deal. Almost all of
that turns out to be one missing curvature term in the pooled model rather than anything about
the profiles. The section is written in that order deliberately, because the wrong answer is the
one that looks convincing.

In [ ]:
import os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.preprocessing import StandardScaler

RS = 42

df = pd.read_csv('../data/clean.csv').merge(pd.read_csv('../data/profiles.csv'), on='runner_id')

# drop-outs have no finish time, so they leave the regression (they had their own section in 03)
d = df[df.dnf == 0].copy()
len(d)

## Setup

`rest_days_per_week` is dropped: it is exactly `min(7 - runs_per_week, 3)`, so in a regression it
is not double weight as it was in the distance metric, it is collinearity.

Predictors are standardised on the full sample, not within profile, so a coefficient means the
same thing in every group and the four columns can be read side by side.

No `statsmodels` here. OLS by least squares with classical standard errors is 8 lines and keeps
the notebook dependency-free.

In [ ]:
BODY = ['age', 'running_experience_months', 'previous_marathon_count', 'weekly_mileage_miles',
        'runs_per_week', 'speed_work_sessions_per_week', 'rest_days_per_week',
        'cross_training_hours_per_week', 'resting_heart_rate_bpm', 'vo2_max', 'bmi',
        'injury_count']
PRED = [c for c in BODY if c != 'rest_days_per_week']

Z = StandardScaler().fit_transform(d[PRED])
y = d.actual_finish_time_minutes.values
tss = ((y - y.mean()) ** 2).sum()


def ols(Xd, y):
    """returns coefficients (intercept first), standard errors, residual sum of squares, n, n_params"""
    A = np.column_stack([np.ones(len(Xd)), Xd])
    b = np.linalg.lstsq(A, y, rcond=None)[0]
    r = y - A @ b
    n, k = A.shape
    cov = (r @ r / (n - k)) * np.linalg.pinv(A.T @ A)
    return b, np.sqrt(np.diag(cov)), r @ r, n, k

## Pooled model

One model for everyone. This is the baseline every later comparison is against.

In [ ]:
b, se, rss_pooled, n, k = ols(Z, y)
print(f'R2 = {1 - rss_pooled / tss:.4f}   (minutes per SD of each predictor)')
pd.DataFrame({'coef': b[1:], 'se': se[1:]}, index=PRED).round(2).sort_values('coef')

Running experience dominates everything: one standard deviation is worth about 18 minutes,
against 3 for mileage and speed work. Injuries cost 5 minutes per additional injury-SD. `bmi` and
`previous_marathon_count` do nothing, consistent with the flat correlations in `03`.

## Slopes fitted separately in each profile

The direct answer to the question, taken at face value.

In [ ]:
def by_profile(Xd, labels=('P0', 'P1', 'P2', 'P3')):
    """fit the same design separately in each profile; returns coefficients and standard errors."""
    co, er, rss, npar = {}, {}, 0.0, 0
    for p in range(4):
        m = (d.profile == p).values
        bp, sp, rp, _, kp = ols(Xd[m], y[m])
        co[labels[p]], er[labels[p]] = bp[1:], sp[1:]
        rss += rp; npar += kp
    return pd.DataFrame(co), pd.DataFrame(er), rss, npar


coef_lin, se_lin, rss_sep, npar_sep = by_profile(Z)
coef_lin.index = se_lin.index = PRED
print(f'separate-slopes R2 = {1 - rss_sep / tss:.4f}')
coef_lin.round(2)

This looks like a substantial finding. An SD of experience is worth 30 minutes to the
low-frequency beginners and 5 minutes to the experienced high-load group. Mileage and speed work
show the same pattern, weaker. Injury cost is flat at about 5 minutes everywhere.

The story writes itself: diminishing returns to training, everything matters less once you are
already good. It is also wrong, or mostly wrong, and the next two cells are why.

## Is it heterogeneity, or a missing term in the pooled model?

Formally the comparison is a Chow test: profile-specific intercepts (levels differ, slopes do
not) against fully profile-specific slopes.

The trap is that a curved relationship fitted with a straight line produces exactly this pattern.
Profiles are ordered on experience by construction, so if finish time flattens out with
experience, each profile sits on a different part of the curve and each recovers a different local
slope. That is curvature in the pooled model showing up as fake heterogeneity.

In [ ]:
def chow(extra_cols):
    """restricted = shared slopes + profile dummies; unrestricted = slopes free per profile."""
    Xd = np.column_stack([Z] + extra_cols)
    D = np.column_stack([(d.profile == p).astype(float) for p in (1, 2, 3)])
    _, _, rss_r, _, k_r = ols(np.column_stack([Xd, D]), y)
    rss_u, npar_u = by_profile(Xd)[2:]
    df1, df2 = npar_u - (k_r + 1), len(y) - npar_u
    F = ((rss_r - rss_u) / df1) / (rss_u / df2)
    return {'restricted_R2': 1 - rss_r / tss, 'unrestricted_R2': 1 - rss_u / tss,
            'delta_R2': (rss_r - rss_u) / tss, 'F': F, 'p': stats.f.sf(F, df1, df2)}


exp = d.running_experience_months.values
exp_z = (exp - exp.mean()) / exp.std()
log_exp = ((np.log(exp + 1) - np.log(exp + 1).mean()) / np.log(exp + 1).std())[:, None]
exp_sq = (exp_z ** 2)[:, None]

specs = {'linear only': [], '+ log(experience)': [log_exp],
         '+ experience²': [exp_sq], '+ both': [log_exp, exp_sq]}
chow_tab = pd.DataFrame({name: chow(cols) for name, cols in specs.items()}).T
chow_tab.round(4)

There it is. With a linear pooled model the profile-specific slopes add 5.9% of variance and the
test is overwhelming. Add a single squared experience term to the pooled model and the same
comparison adds **0.4%**. Roughly 93% of the apparent heterogeneity was one missing curvature term.

The F statistic stays significant, because at 78,000 rows almost anything does. That is why
`delta_R2` is in the table and why it, not the p-value, is the thing being read.

In [ ]:
# the curve that was doing the damage, and where each profile sits on it
bins = pd.qcut(d.running_experience_months, 25, duplicates='drop')
curve = d.groupby(bins, observed=True).agg(exp=('running_experience_months', 'mean'),
                                           finish=('actual_finish_time_minutes', 'mean'))
d.groupby('profile').running_experience_months.agg(['mean', 'std', 'min', 'max']).round(1)

The profiles occupy different stretches of the experience range: mean 21, 28, 33 and 95 months.
Profile 3 sits out on the flat tail of the curve, which is the whole of its shallower slope.

## What actually survives

Refit per profile with the curvature term included, and look at which coefficients still spread
further apart than their standard errors.

In [ ]:
coef_q, se_q, _, _ = by_profile(np.column_stack([Z, exp_sq]))
coef_q.index = se_q.index = PRED + ['experience²']

survive = coef_q.copy()
survive['spread'] = coef_q.max(axis=1) - coef_q.min(axis=1)
survive['largest_se'] = se_q.max(axis=1)
survive.sort_values('spread', ascending=False).round(2)

Two things are real but small.

Resting heart rate costs the high-aerobic-capacity profile about 5.3 minutes per SD against 1.8
elsewhere, and VO2 max changes sign in that group. Within a population already at VO2 max 65 and
resting HR 65, the marginal aerobic variable behaves differently from the rest of the field. That
is at least a plausible physiological story rather than an artefact.

Experience still spreads (-20 in profile 3 against -30 elsewhere) even with the quadratic in, so
the curvature term has not absorbed all of it. A spline would probably take the rest.

Both sit inside a total `delta_R2` of 0.004. Detectable, not important. The honest one-line answer
to the research question is: **the drivers do not meaningfully differ by profile once the pooled
model is specified correctly.**

## Figure

In [ ]:
plt.rcParams.update({'figure.dpi': 150, 'font.size': 8, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.titlesize': 9, 'axes.titleweight': 'bold',
                     'axes.labelsize': 8, 'legend.frameon': False})
INK, ACC, GREY = '#2b2b2b', '#c0392b', '#9aa0a6'
PAL = ['#4c6ef5', '#e8590c', '#2f9e44', '#ae3ec9']
NAMES = ['Low-frequency beginners', 'Frequent short runs', 'High aerobic capacity', 'Experienced high-load']
os.makedirs('../figures', exist_ok=True)

fig, ax = plt.subplots(2, 2, figsize=(11, 7))

# A: the curve, with each profile's mean experience marked
a = ax[0, 0]
a.scatter(curve.exp, curve.finish, s=16, color=INK, zorder=3, label='binned means')
xs = np.linspace(curve.exp.min(), curve.exp.max(), 200)
a.plot(xs, np.polyval(np.polyfit(d.running_experience_months, y, 1), xs),
       color=ACC, lw=1.2, ls='--', label='linear fit')
a.plot(xs, np.polyval(np.polyfit(d.running_experience_months, y, 2), xs),
       color=ACC, lw=1.4, label='quadratic fit')
for p in range(4):
    mx = d.loc[d.profile == p, 'running_experience_months'].mean()
    a.axvline(mx, color=PAL[p], lw=1.2, alpha=0.7)
    a.text(mx, 316, f'P{p}', color=PAL[p], fontsize=7, ha='center')
a.set_title('The curve that faked the heterogeneity')
a.set_xlabel('running_experience_months'); a.set_ylabel('mean finish time (min)')
a.legend(fontsize=6.5, loc='lower left')

# B: the one dominant variable, before and after
a = ax[0, 1]
for p in range(4):
    a.plot([0, 1], [coef_lin.loc['running_experience_months', f'P{p}'],
                    coef_q.loc['running_experience_months', f'P{p}']],
           '-o', color=PAL[p], ms=5, lw=1.2, label=NAMES[p])
a.set_xlim(-0.3, 1.5); a.set_xticks([0, 1]); a.set_xticklabels(['linear spec', '+ experience²'])
a.set_ylabel('minutes per SD of experience')
a.set_title('Experience: the spread halves once curvature is modelled')
a.legend(fontsize=6, loc='center right')

# C and D: everything else, same x-scale
show = ['weekly_mileage_miles', 'runs_per_week', 'speed_work_sessions_per_week',
        'resting_heart_rate_bpm', 'vo2_max', 'injury_count', 'age',
        'cross_training_hours_per_week', 'bmi']
ypos = np.arange(len(show))
lim = (min(coef_lin.loc[show].min().min(), coef_q.loc[show].min().min()) - 0.6,
       max(coef_lin.loc[show].max().max(), coef_q.loc[show].max().max()) + 0.6)
for col, (tbl, title) in enumerate([(coef_lin, 'Linear spec'), (coef_q, 'With experience²')]):
    a = ax[1, col]
    for p in range(4):
        a.scatter(tbl.loc[show, f'P{p}'], ypos + (p - 1.5) * 0.18, s=20, color=PAL[p])
    a.axvline(0, color=INK, lw=0.7)
    a.set_yticks(ypos)
    a.set_yticklabels(show if col == 0 else [], fontsize=6.5)
    a.set_xlim(*lim); a.set_xlabel('minutes per SD')
    a.set_title(f'{title}: the other {len(show)} predictors')

fig.text(0.5, -0.01,
         'profile-specific slopes add '
         f"{chow_tab.loc['linear only', 'delta_R2']:.1%} of variance in the linear spec, "
         f"{chow_tab.loc['+ experience²', 'delta_R2']:.1%} once curvature is included",
         fontsize=8, color=ACC, ha='center')

fig.suptitle('Do the drivers of finish time differ by profile? Mostly no.',
             fontsize=11, fontweight='bold', x=0.02, ha='left', y=0.995)
fig.tight_layout(rect=[0, 0, 1, 0.97])
fig.savefig('../figures/fig3_drivers.png', bbox_inches='tight')

## Save

In [ ]:
chow_tab.round(4).to_csv('../data/chow_test.csv')
coef_q.round(3).to_csv('../data/coefficients_by_profile.csv')
chow_tab.round(4)

For the report: the profiles differ in level, not in mechanism. Runners in every profile get
the same return per additional standard deviation of training, and the flatter slope in the
experienced group is the shape of the experience curve rather than a property of that group.

This is a stronger section than the alternative would have been. Reporting the linear-model result
as evidence of diminishing returns would have been an easy finding, well presented and wrong, and
the thing that catches it is one squared term.